# Neural Network for Classification with One and Two Hidden Layers

This notebook demonstrates how to build and train a neural network for a binary classification task using PyTorch. 

We will create two models: one with a single hidden layer and another with two hidden layers, and compare their performance on a synthetic dataset.

## 1. Importing Necessary Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Generating Synthetic Data
We will use scikit-learn's `make_moons` function to create a synthetic dataset for binary classification.

In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).float().view(-1, 1)

In [ ]:
X,y

In [ ]:
plt.scatter(X[:,0], X[:,1], c=y, cmap=plt.cm.RdYlBu, edgecolors='k')
plt.title("Synthetic Data for Classification")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 3. Model with One Hidden Layer

### Defining the Network Architecture
We define a neural network with two input neurons, one hidden layer with 10 neurons, and one output neuron. We use the ReLU activation function.

In [ ]:
class NetOneHidden(nn.Module):
    def __init__(self):
        super(NetOneHidden, self).__init__()
        self.fc1 = nn.Linear(2, 10)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

### Training the Model
For binary classification, we use `BCEWithLogitsLoss`, which combines a Sigmoid layer and the Binary Cross Entropy loss in one single class. This is more numerically stable than using a plain Sigmoid followed by a BCELoss.

In [ ]:
X_tensor, y_tensor

In [ ]:
model_one_hidden = NetOneHidden()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_one_hidden.parameters(), lr=0.01)

for epoch in range(1000):
    y_pred = model_one_hidden(X_tensor)
    loss = criterion(y_pred, y_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}')

## Convert logits to probabilities, obtain predicted classes, and compute training accuracy.


In [ ]:
def accuracy(y_pred):
    probs = torch.sigmoid(y_pred).detach().numpy().ravel()    # probabilities in [0,1]
    preds = (probs > 0.5).astype(int)                         # binary predictions
    accuracy = (preds == y).mean()                            # y is the numpy array of true labels

    print(f"Epoch (0-based): {epoch}")
    print(f"Final loss: {loss.item():.4f}")
    print(f"Training accuracy: {accuracy*100:.2f}%")

In [ ]:
accuracy(y_pred)

### Visualizing the Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                         np.arange(y_min, y_max, 0.01))
    
    Z = model(torch.from_numpy(np.c_[xx.ravel(), yy.ravel()]).float()).detach().numpy()
    Z = Z.reshape(xx.shape)
    Z = 1 / (1 + np.exp(-Z)) # Apply sigmoid to get probabilities
    Z = (Z > 0.5).astype(int) # Classify
    
    plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdYlBu)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='k')

# Plotting
plot_decision_boundary(model_one_hidden, X, y)
plt.title("Decision Boundary (1 Hidden Layer)")
plt.show()

## 4. Model with Two Hidden Layers

### Defining the Network Architecture

In [ ]:
class NetTwoHidden(nn.Module):
    def __init__(self):
        super(NetTwoHidden, self).__init__()
        self.fc1 = nn.Linear(2, 10)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(10, 5)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(5, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

### Training the Model

In [ ]:
model_two_hidden = NetTwoHidden()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_two_hidden.parameters(), lr=0.01)

for epoch in range(1000):
    y_pred = model_two_hidden(X_tensor)
    loss = criterion(y_pred, y_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/1000], Loss: {loss.item():.4f}')

In [ ]:
accuracy(y_pred)

### Visualizing the Decision Boundary

In [ ]:
plot_decision_boundary(model_two_hidden, X, y)
plt.title("Decision Boundary (2 Hidden Layers)")
plt.show()

plot_decision_boundary(model_one_hidden, X, y)
plt.title("Decision Boundary (1 Hidden Layer)")
plt.show()

### Try experimenting by: 
- Adding more layers or neurons. 
- Changing activation functions (e.g., `Tanh`). 
- Modifying the learning rate or noise level.

## 5. Conclusion
In this notebook, we have built and trained two neural networks for a binary classification task. 

We can see that both models are able to learn the non-linear decision boundary of the moons dataset. 

The model with two hidden layers might be slightly better at capturing the curvature of the data.